# Pima Indians Diabetes — Hybrydowy klasyfikator (PCA + SVM + Random Forest + Stacking)
Ten notebook realizuje zadanie warsztatowe: wczytanie danych, przygotowanie pipeline, **hyper-tuning (GridSearchCV)** oraz ocena modeli (metryki, macierz pomyłek, ROC-AUC, analiza progu).

> Załóżenia: masz plik `diabetes.csv` w tym samym katalogu notebooka (typowa wersja z kolumną celu `Outcome`).


In [ ]:
# === [0] Importy i stałe ===
from __future__ import annotations

import warnings
from dataclasses import dataclass
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression

RANDOM_STATE = 42
TARGET_COL = "Outcome"

warnings.filterwarnings("ignore")


## [1] Wczytanie danych i szybki raport jakości
Wersja Pima często ma problem: **zera w części kolumn oznaczają brak pomiaru** (np. Glucose, BMI).  
Poniżej zrobimy deterministyczną zamianę `0 -> NaN` dla wybranych pól, a imputację medianą w pipeline (bez leakage w CV).


In [ ]:
# === [1] Wczytanie CSV ===
CSV_PATH = "diabetes.csv"   # <- zmień, jeśli plik ma inną nazwę/ścieżkę

df = pd.read_csv(CSV_PATH)
display(df.head())
print("Kolumny:", list(df.columns))

if TARGET_COL not in df.columns:
    raise ValueError(f"Nie znaleziono kolumny celu '{TARGET_COL}'. Dostępne: {list(df.columns)}")

y = df[TARGET_COL].astype(int)
X = df.drop(columns=[TARGET_COL]).apply(pd.to_numeric, errors="coerce")

print("\nLiczba próbek:", X.shape[0])
print("Liczba cech:", X.shape[1])
print("\nRozkład klas:")
display(y.value_counts().sort_index())
display((y.value_counts().sort_index()/len(y)).round(4))


## [2] Preprocessing: 0 jako brak pomiaru + imputacja + skalowanie
- Dla wybranych kolumn zamieniamy `0 -> NaN`
- Potem w pipeline: `SimpleImputer(median)` i `StandardScaler()`


In [ ]:
def build_preprocessor(feature_names: List[str]) -> Tuple[ColumnTransformer, List[str]]:
    zero_as_missing_candidates = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
    zero_as_missing = [c for c in zero_as_missing_candidates if c in feature_names]
    other_cols = [c for c in feature_names if c not in zero_as_missing]

    numeric_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("zero_as_missing_num", numeric_pipeline, zero_as_missing),
            ("plain_num", numeric_pipeline, other_cols),
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )
    return preprocessor, zero_as_missing

def apply_zero_to_nan(X: pd.DataFrame, cols: List[str]) -> pd.DataFrame:
    X2 = X.copy()
    for c in cols:
        X2[c] = X2[c].replace(0, np.nan)
    return X2

feature_names = list(X.columns)
preprocessor, zero_as_missing_cols = build_preprocessor(feature_names)

print("Kolumny (0->NaN):", zero_as_missing_cols)
X_clean = apply_zero_to_nan(X, zero_as_missing_cols)

display(X_clean.isna().sum().sort_values(ascending=False).head(10))


## [3] Podział train/test (stratyfikacja)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_clean, y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE,
)

print("Train:", X_train.shape, "Test:", X_test.shape)
print("Rozkład klas train:", y_train.value_counts().to_dict())
print("Rozkład klas test: ", y_test.value_counts().to_dict())


## [4] Definicje modeli
1. **Random Forest** (baseline)
2. **PCA + SVM** (pipeline)
3. **Stacking**: (PCA+SVM) + (RF) → meta: LogisticRegression


In [ ]:
def make_model_1_rf(preprocessor: ColumnTransformer) -> Pipeline:
    rf = RandomForestClassifier(
        random_state=RANDOM_STATE,
        n_jobs=-1,
        class_weight="balanced",
    )
    return Pipeline(steps=[("prep", preprocessor), ("rf", rf)])

def make_model_2_pca_svm(preprocessor: ColumnTransformer) -> Pipeline:
    svm = SVC(probability=True, random_state=RANDOM_STATE)
    return Pipeline(
        steps=[("prep", preprocessor), ("pca", PCA(random_state=RANDOM_STATE)), ("svm", svm)]
    )

def make_model_3_stacking(preprocessor: ColumnTransformer) -> Pipeline:
    pca_svm = Pipeline(
        steps=[
            ("prep", preprocessor),
            ("pca", PCA(random_state=RANDOM_STATE)),
            ("svm", SVC(probability=True, random_state=RANDOM_STATE)),
        ]
    )
    rf = Pipeline(
        steps=[
            ("prep", preprocessor),
            ("rf", RandomForestClassifier(
                random_state=RANDOM_STATE, n_jobs=-1, class_weight="balanced"
            )),
        ]
    )
    meta = LogisticRegression(max_iter=500, class_weight="balanced", random_state=RANDOM_STATE)

    stack = StackingClassifier(
        estimators=[("pca_svm", pca_svm), ("rf", rf)],
        final_estimator=meta,
        stack_method="predict_proba",
        n_jobs=-1,
        passthrough=False,
        cv=5,
    )
    return Pipeline(steps=[("stack", stack)])

model1 = make_model_1_rf(preprocessor)
model2 = make_model_2_pca_svm(preprocessor)
model3 = make_model_3_stacking(preprocessor)


## [5] GridSearchCV (hyper-tuning)
Używamy **StratifiedKFold** i scoringu **ROC-AUC** (często lepszy niż accuracy w medycznych).


In [ ]:
def grid_search_model(name: str, estimator: Pipeline, param_grid: Dict, X_train, y_train, scoring: str="roc_auc"):
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    gs = GridSearchCV(
        estimator=estimator,
        param_grid=param_grid,
        scoring=scoring,
        cv=cv,
        n_jobs=-1,
        verbose=1,
        refit=True,
    )
    print(f"\n=== GRID SEARCH: {name} | scoring={scoring} ===")
    gs.fit(X_train, y_train)
    print("Best CV score:", round(gs.best_score_, 4))
    print("Best params:", gs.best_params_)
    return gs

param_grid_1 = {
    "rf__n_estimators": [200, 500],
    "rf__max_depth": [None, 4, 8],
    "rf__max_features": ["sqrt", "log2", 0.7],
    "rf__min_samples_leaf": [1, 2, 4],
}
gs1 = grid_search_model("RandomForest", model1, param_grid_1, X_train, y_train)

param_grid_2 = {
    "pca__n_components": [2, 4, 6, 8],
    "svm__kernel": ["rbf", "linear"],
    "svm__C": [0.1, 1, 10],
    "svm__gamma": ["scale", 0.01, 0.1],
}
gs2 = grid_search_model("PCA + SVM", model2, param_grid_2, X_train, y_train)

param_grid_3 = {
    "stack__pca_svm__pca__n_components": [4, 6, 8],
    "stack__pca_svm__svm__kernel": ["rbf"],
    "stack__pca_svm__svm__C": [0.5, 1, 5, 10],
    "stack__pca_svm__svm__gamma": ["scale", 0.01, 0.1],
    "stack__rf__rf__n_estimators": [200, 500],
    "stack__rf__rf__max_depth": [None, 6, 10],
    "stack__final_estimator__C": [0.5, 1, 2],
}
gs3 = grid_search_model("Stacking", model3, param_grid_3, X_train, y_train)


## [6] Ocena na zbiorze testowym
Raport klasyfikacji, macierz pomyłek oraz ROC-AUC.


In [ ]:
def predict_proba_safe(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    if hasattr(model, "decision_function"):
        scores = model.decision_function(X)
        scores = (scores - scores.min()) / (scores.max() - scores.min() + 1e-12)
        return scores
    return model.predict(X).astype(float)

def eval_model(name: str, model, X_test, y_test, threshold: float=0.5):
    proba = predict_proba_safe(model, X_test)
    y_pred = (proba >= threshold).astype(int)

    print(f"\n=== TEST: {name} (threshold={threshold}) ===")
    print(classification_report(y_test, y_pred, digits=4))
    cm = confusion_matrix(y_test, y_pred)
    print("Confusion matrix [[TN FP],[FN TP]]:")
    print(cm)
    print("ROC-AUC:", round(roc_auc_score(y_test, proba), 4))

    return {
        "model": name,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision_pos": precision_score(y_test, y_pred, zero_division=0),
        "recall_pos": recall_score(y_test, y_pred, zero_division=0),
        "f1_pos": f1_score(y_test, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, proba),
    }

m1 = eval_model("RF", gs1.best_estimator_, X_test, y_test)
m2 = eval_model("PCA+SVM", gs2.best_estimator_, X_test, y_test)
m3 = eval_model("STACK", gs3.best_estimator_, X_test, y_test)

summary = pd.DataFrame([m1, m2, m3]).sort_values("roc_auc", ascending=False)
summary


## [7] Analiza progu decyzyjnego (ważne klinicznie)
Często wolimy zmniejszyć FN (false negatives) kosztem FP. Sprawdźmy, jak zmienia się recall/precision dla klasy 1.


In [ ]:
def threshold_sweep(model, X_test, y_test, thresholds):
    proba = predict_proba_safe(model, X_test)
    rows = []
    for t in thresholds:
        y_pred = (proba >= t).astype(int)
        rows.append({
            "threshold": t,
            "precision_pos": precision_score(y_test, y_pred, zero_division=0),
            "recall_pos": recall_score(y_test, y_pred, zero_division=0),
            "f1_pos": f1_score(y_test, y_pred, zero_division=0),
            "accuracy": accuracy_score(y_test, y_pred),
        })
    return pd.DataFrame(rows)

thresholds = [0.2, 0.3, 0.4, 0.5, 0.6]
threshold_sweep(gs3.best_estimator_, X_test, y_test, thresholds)
